# Employee Attrition Prediction — Logistic Regression

Predict whether an employee will leave the organization using Logistic Regression.

In [1]:
import pandas as pd

train = pd.read_csv('../dataset/train.csv')
test = pd.read_csv('../dataset/test.csv')

print(f"Train: {train.shape}, Test: {test.shape}")
train.head()

ModuleNotFoundError: No module named 'pandas'

In [ ]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 59598 entries, 0 to 59597
Data columns (total 24 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Employee ID               59598 non-null  int64
 1   Age                       59598 non-null  int64
 2   Gender                    59598 non-null  str  
 3   Years at Company          59598 non-null  int64
 4   Job Role                  59598 non-null  str  
 5   Monthly Income            59598 non-null  int64
 6   Work-Life Balance         59598 non-null  str  
 7   Job Satisfaction          59598 non-null  str  
 8   Performance Rating        59598 non-null  str  
 9   Number of Promotions      59598 non-null  int64
 10  Overtime                  59598 non-null  str  
 11  Distance from Home        59598 non-null  int64
 12  Education Level           59598 non-null  str  
 13  Marital Status            59598 non-null  str  
 14  Number of Dependents      59598 non-null  int64
 

In [ ]:
train.isnull().sum()

Employee ID                 0
Age                         0
Gender                      0
Years at Company            0
Job Role                    0
Monthly Income              0
Work-Life Balance           0
Job Satisfaction            0
Performance Rating          0
Number of Promotions        0
Overtime                    0
Distance from Home          0
Education Level             0
Marital Status              0
Number of Dependents        0
Job Level                   0
Company Size                0
Company Tenure              0
Remote Work                 0
Leadership Opportunities    0
Innovation Opportunities    0
Company Reputation          0
Employee Recognition        0
Attrition                   0
dtype: int64

In [ ]:
train.duplicated().sum()

np.int64(0)

In [ ]:
train['Attrition'].value_counts()

Attrition
Stayed    31260
Left      28338
Name: count, dtype: int64

In [ ]:
train['Attrition'].value_counts(normalize=True).mul(100).round(1)

Attrition
Stayed    52.5
Left      47.5
Name: proportion, dtype: float64

In [ ]:
for col in train.select_dtypes(include=['object', 'str']).columns:
    train[col] = train[col].str.replace('\u2019', "'", regex=False).str.replace('\u2018', "'", regex=False)
    test[col] = test[col].str.replace('\u2019', "'", regex=False).str.replace('\u2018', "'", regex=False)

In [ ]:
binary_cols = ['Gender', 'Overtime', 'Remote Work', 'Leadership Opportunities', 'Innovation Opportunities']
ordinal_cols = ['Work-Life Balance', 'Job Satisfaction', 'Performance Rating',
                'Education Level', 'Job Level', 'Company Size', 'Company Reputation', 'Employee Recognition']
nominal_cols = ['Job Role', 'Marital Status']
numeric_cols = ['Age', 'Years at Company', 'Monthly Income', 'Number of Promotions',
                'Distance from Home', 'Number of Dependents', 'Company Tenure']

In [ ]:
train[numeric_cols].describe()

,Age,Years at Company,Monthly Income,Number of Promotions,Distance from Home,Number of Dependents,Company Tenure
count,59598.000000,59598.000000,59598.000000,59598.000000,59598.000000,59598.000000,59598.000000
mean,38.565875,15.753901,7302.397983,0.832578,50.007651,1.648075,55.758415
std,12.079673,11.245981,2151.457423,0.994991,28.466459,1.555689,25.411090
min,18.000000,1.000000,1316.000000,0.000000,1.000000,0.000000,2.000000
25%,28.000000,7.000000,5658.000000,0.000000,25.000000,0.000000,36.000000
50%,39.000000,13.000000,7354.000000,1.000000,50.000000,1.000000,56.000000
75%,49.000000,23.000000,8880.000000,2.000000,75.000000,3.000000,76.000000
max,59.000000,51.000000,16149.000000,4.000000,99.000000,6.000000,128.000000


In [ ]:
ordinal_order = {
    'Work-Life Balance': ['Poor', 'Fair', 'Good', 'Excellent'],
    'Job Satisfaction': ['Low', 'Medium', 'High', 'Very High'],
    'Performance Rating': ['Below Average', 'Average', 'High', 'Low'],
    'Education Level': ['High School', 'Associate Degree', "Bachelor's Degree", "Master's Degree", 'PhD'],
    'Job Level': ['Entry', 'Mid', 'Senior'],
    'Company Size': ['Small', 'Medium', 'Large'],
    'Company Reputation': ['Poor', 'Fair', 'Good', 'Excellent'],
    'Employee Recognition': ['Low', 'Medium', 'High', 'Very High']
}

binary_mappings = {
    'Gender': {'Male': 0, 'Female': 1},
    'Overtime': {'No': 0, 'Yes': 1},
    'Remote Work': {'No': 0, 'Yes': 1},
    'Leadership Opportunities': {'No': 0, 'Yes': 1},
    'Innovation Opportunities': {'No': 0, 'Yes': 1}
}

In [ ]:
for col, mapping in binary_mappings.items():
    train[col] = train[col].map(mapping)
    test[col] = test[col].map(mapping)

train[binary_cols].head()

,Gender,Overtime,Remote Work,Leadership Opportunities,Innovation Opportunities
0,0,0,0,0,0
1,1,0,0,0,0
2,1,0,0,0,0
3,1,0,1,0,0
4,0,1,0,0,0


In [ ]:
y_train = train['Attrition'].map({'Stayed': 0, 'Left': 1})
y_test = test['Attrition'].map({'Stayed': 0, 'Left': 1})
X_train = train.drop(columns=['Attrition', 'Employee ID'])
X_test = test.drop(columns=['Attrition', 'Employee ID'])

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}, y_test:  {y_test.shape}")

X_train: (59598, 22), y_train: (59598,)
X_test:  (14900, 22), y_test:  (14900,)


In [ ]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('scaler', StandardScaler())]), numeric_cols),
    ('ordinal', Pipeline([('ordinal', OrdinalEncoder(categories=[ordinal_order[c] for c in ordinal_cols]))]), ordinal_cols),
    ('nominal', Pipeline([('onehot', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))]), nominal_cols),
    ('binary', Pipeline([('scaler', StandardScaler())]), binary_cols)
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)

print(f"Processed: X_train {X_train_proc.shape}, X_test {X_test_proc.shape}")

Processed: X_train (59598, 26), X_test (14900, 26)


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

model = LogisticRegression(C=1.0, solver='lbfgs', random_state=42, max_iter=1000)
model.fit(X_train_proc, y_train)
y_pred = model.predict(X_test_proc)
y_prob = model.predict_proba(X_test_proc)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob):.4f}")

Accuracy:  0.7409
Precision: 0.7303
Recall:    0.7150
F1-Score:  0.7226
ROC-AUC:   0.8305


In [ ]:
import joblib

joblib.dump(model, '../model/model.pkl')
joblib.dump(preprocessor, '../model/preprocessor.pkl')
joblib.dump(binary_mappings, '../model/binary_mappings.pkl')
print("Model, preprocessor, and mappings saved to Model/")

FileNotFoundError: [Errno 2] No such file or directory: '../model/model.pkl'